<a href="https://colab.research.google.com/github/abuqaiselegant/giants-from-dust/blob/main/notebooks/03-gpt/rag-simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A minimal RAG pipeline (from scratch)

**RAG** (Retrieval-Augmented Generation) means: before asking a language model to answer, you **retrieve** relevant text from a knowledge source, **inject** it into the prompt, then **generate** the answer. That way the model can ground responses in *your* documents instead of relying only on what it memorized during training.

This notebook implements a tiny end-to-end version:

1. **Index** — split documents into chunks and build a searchable representation.
2. **Retrieve** — given a user question, score chunks and take the top few.
3. **Augment** — build a prompt that includes those chunks as "context".
4. **Generate** — call an LLM (here we use a **placeholder** so everything runs offline; you can swap in a real API).

Production RAG adds chunking strategies, better embedders (e.g. neural embeddings), vector databases, re-ranking, citation, and evaluation — but the *shape* of the pipeline is the same.

## Why RAG?

- **Fresh / private data** — manuals, wikis, tickets, and policies change; RAG supplies current snippets without retraining the whole model.
- **Reduced hallucination risk** — answers can be tied to retrieved passages (especially if you ask the model to quote or stay within context).
- **Interpretability** — you can show users *which* chunks influenced the answer.

Trade-offs: retrieval quality becomes a bottleneck; very small or ambiguous queries may pull the wrong chunks; context windows limit how much text you can pass in.

## 1. Sample knowledge base

In real systems this would be files, a database, or a crawl. We use a few short "documents" so the logic is easy to follow.

In [ ]:
from __future__ import annotations

import math
import re
from dataclasses import dataclass
from typing import Callable, List, Sequence, Tuple

# Pretend this is your internal documentation
RAW_DOCS: List[str] = [
    "Our refund policy allows full refunds within 14 days of purchase if the item is unused.",
    "After 14 days, we offer store credit only. Shipping fees are non-refundable.",
    "To request a refund, email support@example.com with your order ID and reason.",
    "Premium subscribers get priority support with a 4-hour first response time.",
    "Free tier support responds within 2 business days.",
]

RAW_DOCS

## 2. Chunking

**Chunking** cuts text into pieces that fit in the model context and match retrieval granularity. Here each "document" is already small, so we use one chunk per document. In practice you might split on headings, token counts, or sentence boundaries.

We keep `(chunk_id, text)` tuples so retrieved chunks can be cited or logged.

In [ ]:
def simple_chunks(documents: Sequence[str]) -> List[Tuple[str, str]]:
    """One chunk per document; id is doc index as string for readability."""
    return [(str(i), doc) for i, doc in enumerate(documents)]


CHUNKS = simple_chunks(RAW_DOCS)
CHUNKS

## 3. Retrieval with TF–IDF + cosine similarity

**Retrieval** turns "find text related to this question" into math:

- **Tokenization** — split text into words (here: lowercase alphanumeric tokens).
- **TF** (term frequency) — how often a word appears in a chunk, normalized by the max count in that chunk (simple scaling).
- **IDF** (inverse document frequency) — rare words across the corpus get higher weight so "the" does not dominate.
- **Vector** — for each chunk, one number per vocabulary entry: roughly `TF × IDF`.
- **Query** — same vectorization for the user's question.
- **Score** — **cosine similarity** between query vector and each chunk vector (1 = aligned direction, 0 = orthogonal).

Modern apps often use **dense embeddings** (neural vectors) and **approximate nearest neighbor** search in a vector DB. TF–IDF is older but needs **no ML dependencies** and illustrates the same *retrieve top-k* interface.

In [ ]:
from __future__ import annotations

def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


def cosine_sim(a: Sequence[float], b: Sequence[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    if na == 0 or nb == 0:
        return 0.0
    return dot / (na * nb)


@dataclass
class TFIDFIndex:
    """Tiny in-memory index: chunk_id -> sparse-friendly dense TF-IDF vector."""

    chunk_ids: List[str]
    texts: List[str]
    vocab: List[str]
    word2idx: dict[str, int]
    idf: List[float]
    doc_vectors: List[List[float]]

    @classmethod
    def build(cls, chunks: List[Tuple[str, str]]) -> TFIDFIndex:
        ids = [c[0] for c in chunks]
        texts = [c[1] for c in chunks]
        tokenized = [tokenize(t) for t in texts]

        vocab_set: set[str] = set()
        for toks in tokenized:
            vocab_set.update(toks)
        vocab = sorted(vocab_set)
        word2idx = {w: i for i, w in enumerate(vocab)}

        n_docs = len(texts)
        df = [0] * len(vocab)
        for toks in tokenized:
            for w in set(toks):
                df[word2idx[w]] += 1

        # Smoothed IDF: avoids divide-by-zero and dampens extremes
        idf = [math.log((n_docs + 1) / (d + 1)) + 1.0 for d in df]

        def tfidf_vec(toks: List[str]) -> List[float]:
            vec = [0.0] * len(vocab)
            if not toks:
                return vec
            counts: dict[str, int] = {}
            for t in toks:
                counts[t] = counts.get(t, 0) + 1
            max_tf = max(counts.values())
            for term, c in counts.items():
                if term not in word2idx:
                    continue
                idx = word2idx[term]
                tf = c / max_tf
                vec[idx] = tf * idf[idx]
            return vec

        doc_vectors = [tfidf_vec(toks) for toks in tokenized]
        return cls(ids, texts, vocab, word2idx, idf, doc_vectors)

    def _query_vector(self, query: str) -> List[float]:
        vec = [0.0] * len(self.vocab)
        toks = tokenize(query)
        if not toks:
            return vec
        counts: dict[str, int] = {}
        for t in toks:
            counts[t] = counts.get(t, 0) + 1
        max_tf = max(counts.values())
        for term, c in counts.items():
            if term not in self.word2idx:
                continue
            idx = self.word2idx[term]
            tf = c / max_tf
            vec[idx] = tf * self.idf[idx]
        return vec

    def retrieve(self, query: str, k: int = 3) -> List[Tuple[str, float, str]]:
        q = self._query_vector(query)
        ranked: List[Tuple[str, float, str]] = []
        for cid, dv, text in zip(self.chunk_ids, self.doc_vectors, self.texts):
            ranked.append((cid, cosine_sim(q, dv), text))
        ranked.sort(key=lambda x: x[1], reverse=True)
        return ranked[:k]


index = TFIDFIndex.build(CHUNKS)
index.retrieve("How fast is support for paying customers?", k=2)

## 4. Augmentation — build the LLM prompt

**Augmentation** is where you decide *how* to present retrieved text to the model. A common pattern:

- System or instruction: "Answer using only the context below; if unsure, say you don't know."
- Context: the top-k chunks, often labeled `Source 1`, `Source 2`, …
- User question: the actual query.

Good prompt design reduces made-up facts and makes it easier to audit answers.

In [ ]:
def format_context(retrieved: List[Tuple[str, float, str]]) -> str:
    blocks = []
    for i, (cid, score, text) in enumerate(retrieved, start=1):
        blocks.append(f"[Source {i} | id={cid} | score={score:.3f}]\n{text}")
    return "\n\n".join(blocks)


def build_rag_prompt(question: str, retrieved: List[Tuple[str, float, str]]) -> str:
    ctx = format_context(retrieved)
    return (
        "You are a careful support assistant. Use ONLY the context below. "
        "If the context does not contain the answer, say you do not have enough information.\n\n"
        f"Context:\n{ctx}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )


q = "Can I get my shipping cost back?"
top = index.retrieve(q, k=2)
print(build_rag_prompt(q, top))

## 5. Generation (placeholder vs real LLM)

**Generation** is a call to your chosen model on the prompt above. This notebook does not require API keys:

- `mock_llm` echoes a short answer and shows that the *structure* is what matters for wiring RAG.
- To use a real model, replace `generate` with e.g. OpenAI, Anthropic, or a local `transformers` pipeline — the inputs are always **prompt strings** (or chat messages) built from retrieval + question.

In [ ]:
GenerateFn = Callable[[str], str]


def mock_llm(prompt: str) -> str:
    """Standalone demo: pretend we read the context and answer conservatively."""
    if "non-refundable" in prompt.lower():
        return (
            "According to the context, shipping fees are non-refundable, so you typically "
            "cannot get shipping costs back unless a policy exception is stated."
        )
    if "premium" in prompt.lower() and "support" in prompt.lower():
        return (
            "Premium subscribers get priority support with a first response within about 4 hours."
        )
    return "I do not have enough information in the provided context to answer confidently."


def generate_answer(prompt: str, llm: GenerateFn = mock_llm) -> str:
    return llm(prompt).strip()

## 6. End-to-end `rag_answer`

This function is the full **RAG** loop: **R**etrieve → **A**ugment → **G**enerate.

In [ ]:
def rag_answer(
    question: str,
    *,
    index: TFIDFIndex,
    k: int = 3,
    llm: GenerateFn = mock_llm,
) -> Tuple[str, List[Tuple[str, float, str]]]:
    retrieved = index.retrieve(question, k=k)
    prompt = build_rag_prompt(question, retrieved)
    answer = generate_answer(prompt, llm=llm)
    return answer, retrieved


for question in [
    "Can I get my shipping cost back?",
    "How fast is support if I pay for premium?",
]:
    answer, retrieved = rag_answer(question, index=index, k=2)
    print("Q:", question)
    print("A:", answer)
    print("Used chunks:", [c[0] for c in retrieved])
    print("-" * 60)

## 7. Mental model and next steps

```
Documents  -->  chunks  -->  index (TF-IDF or embeddings)
                                    |
User question  --------------------> retrieve top-k
                                    |
                                    v
                          prompt = instruction + context + question
                                    |
                                    v
                                   LLM  -->  answer
```

**What to add in real systems**

| Piece | Typical upgrade |
|--------|------------------|
| Index | Embedding model + FAISS / pgvector / hosted vector store |
| Chunks | Overlapping windows, metadata filters (product, date, ACL) |
| Retrieval | Hybrid search (keyword + vector), re-ranker |
| Safety | Grounding checks, citation of source ids, refusal when score is low |

You now have a **minimal, readable RAG** skeleton: swap the retriever and the `llm` callable while keeping the same pipeline shape.